<a href="https://colab.research.google.com/github/xconer-svg/colab-seminar/blob/main/spaceship_titanic_knn_seminar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 1. EDA и KNN на Spaceship Titanic

На этом семинаре мы решим задачу **бинарной классификации**: по данным о пассажире предскажем, был ли он перенесён в другое измерение (`Transported`).

# File and Data Field Descriptions
### train.csv - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.
- PassengerId - A unique Id for each passenger. Each Id takes the form gggg_pp where gggg indicates a group the passenger is travelling with and pp is their number within the group. People in a group are often family members, but not always.
- HomePlanet - The planet the passenger departed from, typically their planet of permanent residence.
- CryoSleep - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
- Cabin - The cabin number where the passenger is staying. Takes the form deck/num/side, where side can be either P for Port or S for Starboard.
- Destination - The planet the passenger will be debarking to.
- Age - The age of the passenger.
- VIP - Whether the passenger has paid for special VIP service during the voyage.
- RoomService, FoodCourt, ShoppingMall, Spa, VRDeck - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
- Name - The first and last names of the passenger.
- Transported - Whether the passenger was transported to another dimension. This is the target, the column you are trying to predict.
### test.csv
- Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set.
### sample_submission.csv - A submission file in the correct format.
- PassengerId - Id for each passenger in the test set.
- Transported - The target. For each passenger, predict either True or False.

## 1. Импорт библиотек

Обычно все импорты собирают в начале ноутбука. Сокращения `np`, `pd`, `plt` и `sns` общеприняты.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statistics as stats

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 2. Знакомство с NumPy

`numpy` удобен для быстрых вычислений над массивами. Возьмём пример из лекции и посчитаем меры центральной тенденции и разброса.

In [ ]:
# TODO: создайте numpy-массив numbers из списка [4, 8, 6, 5, 3, 100, 7, 6] с помощью np.array()


In [ ]:
# TODO: с помощью np.unique(numbers, return_counts=True) получите уникальные значения и их количество (values, counts)
аааа

In [ ]:
# TODO: найдите моду — значение с максимальной частотой (np.argmax(counts))


In [ ]:
# TODO: выведите на экран массив, его размер, среднее, медиану, моду, дисперсию,
# стандартное отклонение, размах и IQR (np.mean, np.median, mode, np.var, np.std, max-min, np.percentile, IQR)


In [ ]:
# TODO: с помощью np.where пометьте значения > 10 как "большое", остальные — как "обычное"


Большое значение `100` заметно сдвигает среднее, но гораздо слабее влияет на медиану. Поэтому для скошенных распределений и данных с выбросами пропуски часто заполняют медианой.

## 3. Загружаем и читаем данные с pandas

`pd.read_csv()` превращает CSV-файл в `DataFrame` -таблицу pandas.

In [ ]:
train = pd.read_csv("train.csv", sep = ",",
                    encoding = "utf-8", skipinitialspace = True)
test = pd.read_csv("test.csv", sep = ",",
                   encoding = "utf-8", skipinitialspace = True)

print("Размер train:", train.shape)
print("Размер test: ", test.shape)

In [ ]:
# TODO: посмотрите на первые строки train с помощью head()


### Что лежит в таблице

- строка - один пассажир;
- признаки (`X`) - сведения о пассажире;
- целевая переменная (`y`) - `Transported`;
- в `test` нет `Transported`: его и должна предсказать модель.

In [ ]:
# TODO: выведите список столбцов train и столбцы, которых нет в test


## 4. Смотрим на данные

In [ ]:
# TODO: посмотрите на типы столбцов и количество непустых значений с помощью info()


In [ ]:
# TODO: посчитайте основные статистики числовых столбцов с помощью describe()


посмотрим на количество пропусков в каждой колонке

In [ ]:
# TODO: посчитайте количество пропусков в каждой колонке с помощью isna().sum()


теперь сделаем то же самое, но более витьеватым способом

In [ ]:
# TODO: соберите таблицу missing_count / missing_percent и отсортируйте по убыванию пропусков
missing = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_percent": train.isna().mean().mul(100).round(2),
}).sort_values("missing_count", ascending=False)

missing

In [ ]:
# TODO: проверьте дубликаты строк и PassengerId, посмотрите баланс целевой переменной Transported


Классы сбалансированы: объектов `True` и `False` примерно поровну. Для нас это очень хорошо. Модель не будет склонна предсказывать только определенный класс. Например: у нас сильный дисбаланс и  класса 1 5% ,а класса 0 95%. Модель может послать класс 1 далеко и надолго и постоянно предсказывать 0, тогда точность будет 95%. Казалось бы, чем нас не устраивает эта метрика. Но в наших интересах предсказывать и класс 1 тоже.

## 5. Matplotlib

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

train["Transported"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0, 0], color=["tomato", "seagreen"]
)
axes[0, 0].set_title("Баланс целевой переменной")
axes[0, 0].set_xlabel("Transported")
axes[0, 0].set_ylabel("Количество пассажиров")
axes[0, 0].tick_params(axis="x", rotation=0) # настраивает внешний вид записей

axes[0, 1].hist(train["Age"].dropna(), bins=30, color="cornflowerblue", edgecolor="white")
axes[0, 1].set_title("Распределение возраста")
axes[0, 1].set_xlabel("Возраст")
axes[0, 1].set_ylabel("Количество пассажиров")

train.boxplot(column="Age", by="Transported", ax=axes[1, 0])
axes[1, 0].set_title("Возраст по классам")
axes[1, 0].set_xlabel("Transported")
axes[1, 0].set_ylabel("Возраст")

homeplanet_rate = train.groupby("HomePlanet", dropna=False)["Transported"].mean().sort_values() # вычисляем долю успешно перемещенных на каждой планете
homeplanet_rate.plot(kind="bar", ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Доля Transported по HomePlanet")
axes[1, 1].set_xlabel("HomePlanet")
axes[1, 1].set_ylabel("Доля Transported")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.suptitle("")
plt.tight_layout() # предотвращает наложение графов друг на друга
plt.show()

In [ ]:
# TODO: постройте распределение возраста через sns.histplot с kde


In [ ]:
# TODO: постройте barplot для homeplanet_rate


In [ ]:
# loc и iloc
# loc - Использует реальные названия строк (индексы) и заголовки столбцов
# iloc - Использует порядковые номера элементов, начиная с нуля

# TODO: выведите первые 9 строк и колонки "PassengerId", "VIP", "Transported" с помощью loc


In [ ]:
# TODO: выведите 3-ю строку по первой колонке с помощью iloc


In [ ]:
# TODO: выведите пассажиров старше 30 лет с помощью loc


In [ ]:
# TODO: выведите пассажиров младше 40, которые потратили больше 5000 на еду (комбинация условий)


### Расходы пассажиров

Создадим новый признак: общую сумму расходов по пяти сервисам корабля. `sum(axis=1)` складывает значения по строке.

In [ ]:
# TODO: создайте признак TotalSpend — сумму расходов по RoomService, FoodCourt, ShoppingMall, Spa, VRDeck
# (для train_eda и test), выведите train_eda.head()


In [ ]:
# TODO: постройте гистограммы TotalSpend целиком и без верхнего 1% (по 95-му процентилю)


видим, что в основном пассажиры стараются не тратиться, предположим, у них у всех нет статуса VIP. Давайте это проверим

In [ ]:
# TODO: создайте бинарный признак is_spends — тратил ли пассажир что-то вообще (для train_eda и test)


In [ ]:
# TODO: проверьте, есть ли VIP-пассажиры, которые ничего не потратили


мы были не совсем правы
всё-таки, есть випы, которые ничего не потратили

## 6. Тепловая карта корреляций в seaborn

In [ ]:
# Корреляция измеряет линейную связь числовых признаков.
# Булевый target временно превращаем в 0 и 1.

# TODO: соберите numeric_for_corr — числовые столбцы train_eda плюс Transported, переведённый в int


In [ ]:
# TODO: постройте тепловую карту корреляций с помощью sns.heatmap


Важно: корреляция не показывает причинно-следственную связь, также мы ничего не можем сказть про зависимость в целом. Корреляция может сообщить нам только о линейной зависимости признаков. Если признаки зависят, допустим, по косинусу, то корреляция нам об этом ничего не скажет. Кроме того, карта не показывает категориальные признаки, пока мы не перевели их в числа.

## 7. Preprocessing и Feature Engeneering

KNN сравнивает объекты по расстояниям, поэтому:

1. пропуски нужно заполнить;
2. категориальные признаки нужно закодировать числами;
3. числовые признаки нужно привести к сопоставимому масштабу, то есть нормализовать.

Из `Cabin` вида `deck/num/side` возьмём палубу и сторону
Из `PassengerId` вида `gggg_pp` возьмем группу

In [ ]:
# TODO: напишите функцию make_features(df):
#   - добавьте TotalSpend;
#   - разберите Cabin на CabinDeck и CabinSide;
#   - достаньте PassengerGroup из PassengerId;
#   - удалите PassengerId, Cabin, Name.
# Примените её к train_eda (без Transported) и test, получите X, y, X_test, выведите их размеры и X.head()


## 8. Разделяем данные на обучение и валидацию

In [ ]:
# TODO: разбейте X, y на train/valid с помощью train_test_split
# (test_size=0.2, random_state=RANDOM_STATE, stratify=y)


`stratify=y` сохраняет примерно одинаковую долю классов в обеих частях. Валидацию модель не видит во время обучения - на ней мы честно оцениваем качество и выбираем `k`.

## 9. Preprocessing и KNN в Pipeline

Pipeline - программный конвейер, который объединяет несколько операций в одну общую цепочку

SimpleImputer() помогает заполнять нам пропуски( strategy = mean/median/most_freaquent/constant задается отдельно параметром fill_value)

OneHotEncoder() - энкодер, который помогает преобразовывать категориальные признаки в числовые

In [ ]:
# TODO: определите numeric_features и categorical_features по типам столбцов X


In [ ]:
# TODO: соберите numeric_transformer (SimpleImputer + StandardScaler),
# categorical_transformer (SimpleImputer + OneHotEncoder), объедините их в preprocessor
# через ColumnTransformer и соберите knn_pipeline (preprocessor + KNeighborsClassifier)


`Pipeline` важен не только для удобства. Все медианы, моды и параметры масштабирования вычисляются **только на обучающей части**, поэтому информация из валидации не подглядывается.

## 10. Выбираем число соседей k

In [ ]:
# TODO: переберите k из k_values, обучите knn_pipeline для каждого k
# и посчитайте accuracy на train и valid, соберите результат в scores


In [ ]:
# TODO: найдите лучшее k по максимальной validation_accuracy
# и постройте графики accuracy от k для train и valid
best_row = scores.loc[scores["validation_accuracy"].idxmax()]
best_k = int(best_row["k"])

fig, axes = plt.subplots(1, 2, figsize = (15, 5))
axes[0].plot(scores["k"], scores["validation_accuracy"], marker="o")
axes[0].axvline(best_k, color="tomato", linestyle="--", label=f"лучшее k = {best_k}")
axes[0].set_xlabel("Число соседей k")
axes[0].set_ylabel("Accuracy на валидации")
axes[0].set_title("Подбор гиперпараметра k, график валидации")
axes[0].legend()

axes[1].plot(scores["k"], scores["train_accuracy"], marker="o")
axes[1].axvline(best_k, color="tomato", linestyle="--", label=f"лучшее k = {best_k}")
axes[1].set_xlabel("Число соседей k")
axes[1].set_ylabel("Accuracy на трейне")
axes[1].set_title("Подбор гиперпараметра k, график трейна")

plt.tight_layout()
plt.show()

print(f"Лучшее k: {best_k}")
print(f"Лучшая validation accuracy: {best_row['validation_accuracy']:.4f}")

что мы затронем в программе курса

In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(max_depth = 5, eta = 4e-3, n_estimators = 1000)),
])

In [ ]:
xgb_pipeline.fit(X_train, y_train)
preds = xgb_pipeline.predict(X_valid)
score = accuracy_score(preds, y_valid)
score

ничего не выбирали, не тюнили, параметры, проставленные просто так, дали скор лучше, чем knn после перебора k соседей

## 11. Оцениваем выбранную модель

In [ ]:
# TODO: обучите knn_pipeline с лучшим k на train и оцените accuracy на валидации


## 12. Обучаем финальную KNN и создаём submission.csv

In [ ]:
# TODO: обучите финальную модель на всех данных (X, y), получите предсказания для test
# и сохраните их в submission.csv
knn_pipeline.fit(X, y)
test_predictions = knn_pipeline.predict(X_test).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_predictions,
})

submission.to_csv("submission.csv", index=False)
print("Сохранён файл submission.csv размера", submission.shape)
submission.head()

## Итоги

Мы прошли полный первый ML-пайплайн:

1. загрузили и исследовали данные;
2. нашли пропуски и посмотрели распределения;
3. создали простые признаки;
4. разделили данные на train и validation;
5. заполнили пропуски, закодировали категории и масштабировали числа;
6. подобрали гиперпараметр `k`;
7. оценили KNN;
8. обучили финальную модель и сохранили предсказания.